# Legal RAG — Agentic Pipeline (Kaggle)
Full agentic RAG: retrieval + Qwen3.5-4B LLM + reranker.
Khong can API key — tat ca chay offline.

**Dataset:** `ai-rag-legal-assets` (Public)
- `corpus.zip`: corpus.jsonl (4.1 GB)
- `indexes.zip`: dense.index + BM25 (4.9 GB)
- `src.zip`: ma nguon project

In [ ]:
# Cell 1: Install deps
import subprocess, sys
deps = [
    'transformers>=4.48.0', 'accelerate>=1.3.0',
    'bitsandbytes>=0.43.0', 'sentencepiece>=0.2.0',
    'datasets>=2.19.0', 'faiss-gpu>=1.9.0',
    'bm25s>=0.3.0', 'aiohttp>=3.10.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + deps)
print('Deps installed')

In [ ]:
# Cell 2: Extract assets
import os, sys, zipfile, json, re, time
from pathlib import Path

DATASET = Path('/kaggle/input/ai-rag-legal-assets')
WORKING = Path('/kaggle/working')

for zname in ('corpus.zip', 'indexes.zip', 'src.zip'):
    with zipfile.ZipFile(DATASET / zname) as z:
        z.extractall(WORKING)
    print(f'Extracted {zname}')

os.environ['HF_HOME'] = str(WORKING / 'hf_cache')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
sys.path.insert(0, str(WORKING))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
print('Ready')

In [ ]:
# Cell 3: Load corpus + indexes
t0 = time.time()
from src.data.loading import load_corpus
docs = load_corpus(
    cache_path=str(WORKING / 'data/processed/corpus.jsonl'),
    force_rebuild=False
)
print(f'Corpus: {len(docs)} docs ({time.time()-t0:.1f}s)')

t0 = time.time()
from src.retrieval.indexing import SparseIndex, DenseIndex
si = SparseIndex(); si.load(str(WORKING / 'data/indexes/sparse'))
print(f'BM25: {len(si.doc_ids)} docs ({time.time()-t0:.1f}s)')

di = DenseIndex(); di.load(str(WORKING / 'data/indexes/dense.index'))
print(f'FAISS: {di.index.ntotal} vectors ({time.time()-t0:.1f}s)')

In [ ]:
# Cell 4: Load models (Harrier + Reranker + Qwen3.5-4B)
t0 = time.time()
from src.embedding.harrier_embedding import HarrierEmbedding
embedder = HarrierEmbedding(device='cuda')
print(f'Harrier: {time.time()-t0:.1f}s')

t0 = time.time()
from src.reranker.cross_encoder import CrossEncoderReranker
ce = CrossEncoderReranker(device='cuda')
print(f'Reranker: {time.time()-t0:.1f}s')

t0 = time.time()
from src.llm.hf_client import HFClient
llm = HFClient(device='cuda')
print(f'Qwen3.5-4B: {time.time()-t0:.1f}s')

In [ ]:
# Cell 5: Build pipeline + load evaluation set
from src.pipeline.orchestrator import LegalRAGPipeline
from src.reranker.cross_encoder import LLMReranker, TwoStageReranker

reranker = TwoStageReranker(ce, LLMReranker(llm))
pipeline = LegalRAGPipeline(
    docs=docs, dense_index=di, sparse_index=si,
    embedder=embedder, llm=llm, reranker=reranker,
)

# Load PBGDPL evaluation set
from datasets import load_dataset
ds = load_dataset('tmquan/pbgdpl-vn-legal-qna', split='train', streaming=True)
eval_set = []
for row in ds:
    q = row.get('question_text', row.get('question', ''))
    a = row.get('answer_text', row.get('answer', ''))
    if not q or not a:
        continue
    ref_articles = list(set(re.findall(r'\u0110i\u1ec1u\s+(\d+)', str(a))))
    if len(ref_articles) >= 2:
        eval_set.append({'question': q, 'reference_articles': ref_articles})
    if len(eval_set) >= 50:
        break

print(f'Eval set: {len(eval_set)} queries')
for e in eval_set[:2]:
    print(f'  Q: {e["question"][:50]}... -> {e["reference_articles"][:3]}')

In [ ]:
# Cell 6: Run evaluation
from src.evaluation.metrics import compute_f2_macro, compute_retrieval_metrics

predictions = []
for i, item in enumerate(eval_set):
    q = item['question']
    print(f'\n[{i+1}/{len(eval_set)}] {q[:60]}...')
    try:
        r = await pipeline.answer_agentic(q)
        pred_articles = list(set(re.findall(r'\u0110i\u1ec1u\s+(\d+)', r.final_answer)))
        predictions.append({
            'query': q,
            'predicted': pred_articles,
            'ground_truth': item['reference_articles'],
            'answer': r.final_answer,
            'confidence': r.confidence,
        })
        print(f'  GT: {item["reference_articles"][:3]} | Pred: {pred_articles[:3]} | Conf: {r.confidence:.3f}')
    except Exception as e:
        logger.error(f'Error: {q[:60]} -> {e}')
        predictions.append({'query': q, 'predicted': [], 'ground_truth': item['reference_articles'], 'error': str(e)})

print(f'\n{"="*50}')
print('Computing metrics...')

valid = [p for p in predictions if 'error' not in p]
y_true = [p['ground_truth'] for p in valid]
y_pred = [p['predicted'] for p in valid]

f2 = compute_f2_macro(y_true, y_pred)
ret = compute_retrieval_metrics(y_true, y_pred, k_values=[5, 10, 20, 50])

results = {**f2, **ret, 'num_queries': len(y_true), 'num_errors': len(predictions) - len(y_true)}

print(f'{"="*50}')
print(f'EVALUATION ({len(y_true)} queries)')
print(f'{"="*50}')
print(f'Macro-F2:  {results["macro_f2"]:.4f}')
print(f'Micro-F2:  {results["micro_f2"]:.4f}')
print(f'Micro-Recall: {results["micro_recall"]:.4f}')
for k in [5, 10, 20, 50]:
    print(f'Recall@{k}: {results.get(f"recall@{k}", 0):.4f}')
print(f'{"="*50}')

# Save
with open(WORKING / 'eval_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
with open(WORKING / 'eval_detail.json', 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2, default=str)
print(f'Saved to /kaggle/working/')